# 03 SHAP Token Attributions

Generates token-level SHAP explanations for the fine-tuned MentalBERT model.

Outputs:
- `results/figures/shap/` — one PNG per example (paper figures)
- Interactive HTML plots inline (for exploration)

Expected runtime: ~10–20 min for 32 examples on GPU.

In [ ]:
import sys
sys.path.insert(0, '..')

import shap
import torch
import numpy as np

from src.data import load_splits
from src.shap_explain import (
    load_model, get_predictions, select_examples,
    make_predict_fn, run_shap, save_figures, ID2LABEL,
)

print('CUDA available:', torch.cuda.is_available())

f:\Anaconda3\envs\capstone\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True


## Config
Change `RUN_DIR` and `MODEL_NAME` to switch to RoBERTa.

In [ ]:
RUN_DIR     = '../results/runs/mentalbert'
MODEL_NAME  = 'mental/mental-bert-base-uncased'
MAX_LENGTH  = 256
FIGURES_DIR = '../results/figures/shap'

N_CORRECT = 6
N_WRONG   = 2

## 1 — Load model and data

In [4]:
model, tokenizer, device = load_model(RUN_DIR, MODEL_NAME)
_, _, test_df = load_splits()
print(f'Test set: {len(test_df)} rows  |  device: {device}')

18:12:40  WARNING   src.shap_explain — trainer_state.json not found — using last checkpoint: ..\results\runs\mentalbert\checkpoint-2084
18:12:40  INFO      src.shap_explain — Loading checkpoint: ..\results\runs\mentalbert\checkpoint-2084  →  device=cuda
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6893.73it/s]


Test set: 1041 rows  |  device: cuda


## 2 — Inference on full test set

In [5]:
preds, probs = get_predictions(
    model, tokenizer, test_df['text'].tolist(), device, max_length=MAX_LENGTH,
)

accuracy = (preds == test_df['label_id'].values).mean()
print(f'Test accuracy: {accuracy:.4f}')

for cls, name in ID2LABEL.items():
    mask = test_df['label_id'].values == cls
    print(f'  {name:<12} recall={(preds[mask]==cls).mean():.3f}  n={mask.sum()}')

Test accuracy: 0.7570
  minimum      recall=0.843  n=572
  mild         recall=0.654  n=286
  moderate     recall=0.585  n=118
  severe       recall=0.769  n=65


## 3 — Select curated examples

`N_CORRECT` high-confidence correct + `N_WRONG` borderline failures per class.
Failures are the most interesting for the paper.

In [6]:
selected    = select_examples(test_df, preds, probs, n_correct=N_CORRECT, n_wrong=N_WRONG)
all_examples = [ex for cls_exs in selected.values() for ex in cls_exs]
all_texts    = [ex['text'] for ex in all_examples]

print(f'Total examples selected: {len(all_examples)}')
for cls, name in ID2LABEL.items():
    print(f'  {name}: {len(selected[cls])}')

18:13:07  INFO      src.shap_explain —   minimum: 6 correct + 2 wrong selected
18:13:07  INFO      src.shap_explain —   mild: 6 correct + 2 wrong selected
18:13:07  INFO      src.shap_explain —   moderate: 6 correct + 2 wrong selected
18:13:07  INFO      src.shap_explain —   severe: 6 correct + 2 wrong selected


Total examples selected: 32
  minimum: 8
  mild: 8
  moderate: 8
  severe: 8


## 4 — Run SHAP

Slow step (~10–20 min on GPU). A progress bar appears below.

In [7]:
predict_fn  = make_predict_fn(model, tokenizer, device, max_length=MAX_LENGTH)
shap_values = run_shap(all_texts, predict_fn, tokenizer)

18:13:09  INFO      src.shap_explain — Running SHAP on 32 texts (this may take several minutes) ...
PartitionExplainer explainer: 33it [00:33,  1.26s/it]                        


## 5 — Save paper figures

One PNG per example → `results/figures/shap/`

In [9]:
save_figures(shap_values, all_examples, output_dir=FIGURES_DIR)

18:14:17  INFO      src.shap_explain — Saved 32 figures → ..\results\figures\shap


In [ ]:

# ── Save examples metadata for src/interp_rubric.py ─────────────────────────
# Saves text, labels, test-set indices, and top SHAP tokens for each example.
# Required before running: python src/interp_rubric.py generate
import json
from pathlib import Path

meta = []
for i, ex in enumerate(all_examples):
    pred_cls     = list(ID2LABEL.values()).index(ex["pred_label"])
    sv           = shap_values[i, :, pred_cls]
    tokens       = shap_values.data[i]

    keep = [
        j for j, t in enumerate(tokens)
        if t not in ("[CLS]", "[SEP]", "<s>", "</s>", "<pad>", "[PAD]")
    ]
    sv_vals      = np.array([sv.values[j] for j in keep])
    tokens_clean = [tokens[j]              for j in keep]

    top_idx      = np.argsort(np.abs(sv_vals))[-15:][::-1]
    top_info     = [(tokens_clean[j], round(float(sv_vals[j]), 4)) for j in top_idx]

    fname = f"{i:03d}_{ex['true_label']}_pred{ex['pred_label']}.png"
    meta.append({
        "fig_index":       i,
        "figure_file":     fname,
        "idx":             ex["idx"],
        "text":            ex["text"],
        "true_label":      ex["true_label"],
        "pred_label":      ex["pred_label"],
        "confidence":      ex["confidence"],
        "correct":         ex["correct"],
        "source":          ex["source"],
        "shap_top_tokens": top_info,
    })

meta_path = Path(FIGURES_DIR) / "examples_meta.json"
meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved metadata for {len(meta)} examples -> {meta_path}")


## 6 — Interactive HTML gallery

Red tokens push toward the predicted class; blue push away.

In [10]:
for cls, name in ID2LABEL.items():
    idx = next(i for i, ex in enumerate(all_examples) if ex['true_label'] == name)
    ex  = all_examples[idx]
    print(f'\n── {name} | pred={ex["pred_label"]} conf={ex["confidence"]:.2f} ──')
    print(ex['text'][:200])
    shap.plots.text(shap_values[idx])


── minimum | pred=minimum conf=1.00 ──
Can Washington get their own channel as well as Dallas I’m tired of seeing them niggas on the tv. Baltimore vs chargers today @ 1 n I can’t watch it without red zone



── mild | pred=mild conf=0.99 ──
i hate my depressed ass for this oh God



── moderate | pred=moderate conf=0.98 ──
I stop taking my meds hen i gt depressed because I hate myself and it makes everything worse



── severe | pred=severe conf=0.99 ──
Please help i am again in depression suicidal thoughts


## 7 — Beeswarm (optional)

Overview of which tokens matter most across all examples for the severe class.

In [ ]:
shap.plots.beeswarm(shap_values[:, :, 3])